In [0]:
import datetime as dt
import json
import pathlib
import shutil
import time

import pandas as pd
import requests

API = "https://archive-api.open-meteo.com/v1/archive"
VARIABLES = ["temperature_2m_max", "temperature_2m_min", "precipitation_sum", "relative_humidity_2m_mean"]
MODELO = "era5_land"  # reanálisis ERA5-Land: celdas de 0,1° (unos 11 km), el mismo modelo en toda la serie
DESDE_ANIO = 2015
ZONA = "America/Guayaquil"  # los días van de medianoche a medianoche, hora de Ecuador continental
CANTONES = "/Volumes/vigia/bronze/landing/geografia/cantones.csv"
LANDING = pathlib.Path("/Volumes/vigia/bronze/landing/clima")
POR_LOTE = 25         # cantones por pedido
PRESUPUESTO = 4500    # llamadas por corrida: el plan gratuito permite 600 por minuto, 5000 por hora y 10 000 por día
POR_MINUTO = 500
AGENTE = {"User-Agent": "vigia-dengue/0.1 (proyecto de portafolio sin fines comerciales)"}


class LimiteDeOpenMeteo(Exception):
    """Open-Meteo respondió 429: se acabó el cupo del minuto, la hora o el día."""


def peso(n_cantones, desde, hasta):
    """Cuántas llamadas cuenta Open-Meteo: cada lugar, cada 14 días y cada 10 variables cuentan una."""
    return n_cantones * ((hasta - desde).days + 1) / 14 * len(VARIABLES) / 10


def ultimo_sabado_disponible(hoy):
    """ERA5-Land llega con unos 5 días de retraso: el último sábado con datos completos."""
    dia = hoy - dt.timedelta(days=5)
    return dia - dt.timedelta(days=(dia.weekday() + 2) % 7)  # weekday(): lunes 0 sábado 5


In [0]:
def pedir(lote, desde, hasta):
    parametros = {
        "latitude": ",".join(f"{x:.5f}" for x in lote["latitud"]),
        "longitude": ",".join(f"{x:.5f}" for x in lote["longitud"]),
        "start_date": desde.isoformat(), "end_date": hasta.isoformat(),
        "daily": ",".join(VARIABLES), "timezone": ZONA, "models": MODELO,
    }
    respuesta = requests.get(API, params=parametros, headers=AGENTE, timeout=300)
    if respuesta.status_code == 429:
        raise LimiteDeOpenMeteo(respuesta.text[:300])
    if respuesta.status_code != 200:
        raise RuntimeError(f"Open-Meteo respondió {respuesta.status_code}: {respuesta.text[:300]}")
    datos = respuesta.json()
    datos = datos if isinstance(datos, list) else [datos]  # con un solo lugar no viene en una lista
    if len(datos) != len(lote):
        raise RuntimeError(f"pedí {len(lote)} lugares y llegaron {len(datos)}")
    return datos


cantones = (pd.read_csv(CANTONES, dtype={"dpa_canton": str, "dpa_provin": str})
            .sort_values("dpa_canton").reset_index(drop=True))
try:
    prueba = pedir(cantones[cantones["dpa_canton"] == "0901"], dt.date(2024, 3, 3), dt.date(2024, 3, 9))[0]
    print(f"Guayaquil: celda ({prueba['latitude']}, {prueba['longitude']}), elevación {prueba['elevation']} m")
    display(pd.DataFrame(prueba["daily"]))
except LimiteDeOpenMeteo as error:  # en una corrida programada, el cupo puede estar agotado
    print(f"Open-Meteo pidió esperar: {error}")


In [0]:
def escribir(archivo, lote, respuestas):
    """Un archivo JSON por lote: una línea por cantón, con su código y la respuesta de Open-Meteo"""
    descargado_en = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
    lineas = [json.dumps({"dpa_canton": c.dpa_canton, "latitud_pedida": c.latitud, "longitud_pedida": c.longitud,
                          "latitud_celda": r["latitude"], "longitud_celda": r["longitude"],
                          "elevacion": r.get("elevation"), "modelo": MODELO, "descargado_en": descargado_en,
                          "daily": r["daily"]})
              for c, r in zip(lote.itertuples(), respuestas)]
    archivo.parent.mkdir(parents=True, exist_ok=True)
    parcial = archivo.parent / f"{archivo.name}.parcial"
    parcial.write_text("\n".join(lineas) + "\n", encoding="utf-8")
    shutil.move(parcial, archivo)


lotes = [cantones.iloc[i:i + POR_LOTE] for i in range(0, len(cantones), POR_LOTE)]
hoy = dt.date.today()
ultimo = ultimo_sabado_disponible(hoy)
pendientes = []
for anio in range(DESDE_ANIO, ultimo.year + 1):
    desde, hasta = dt.date(anio, 1, 1), min(dt.date(anio, 12, 31), ultimo)
    for i, lote in enumerate(lotes):
        # La carpeta hasta= cambia una vez por semana en el año en curso: entonces se vuelve a bajar
        archivo = LANDING / f"anio={anio}" / f"hasta={hasta}" / f"lote={i:02d}.json"
        if not archivo.exists():
            pendientes.append((desde, hasta, lote, archivo))

gastado, bajados, motivo = 0.0, 0, "no queda nada pendiente"
for desde, hasta, lote, archivo in pendientes:
    costo = peso(len(lote), desde, hasta)
    if gastado + costo > PRESUPUESTO:
        motivo = f"se usó el presupuesto de esta corrida ({PRESUPUESTO} llamadas)"
        break
    try:
        respuestas = pedir(lote, desde, hasta)
    except LimiteDeOpenMeteo as error:
        motivo = f"Open-Meteo pidió esperar: {error}"
        break
    escribir(archivo, lote, respuestas)
    gastado, bajados = gastado + costo, bajados + 1
    time.sleep(costo / POR_MINUTO * 60)  # para no pasar de POR_MINUTO llamadas por minuto

faltan = len(pendientes) - bajados
costo_faltante = sum(peso(len(l), d, h) for d, h, l, _ in pendientes[bajados:])
print(f"Bajados {bajados} lotes ({gastado:.0f} llamadas). Faltan {faltan} lotes ({costo_faltante:.0f} llamadas). "
      f"Se detuvo porque {motivo}.")


In [0]:
archivos = [{"anio": int(p.parent.parent.name.split("=")[1]), "hasta": p.parent.name.split("=")[1]}
            for p in LANDING.glob("anio=*/hasta=*/lote=*.json")]
if archivos:
    # Un año está completo cuando su último corte tiene los 9 lotes
    display(pd.DataFrame(archivos).groupby(["anio", "hasta"]).size().rename("lotes").reset_index())
else:
    print(f"Todavía no hay archivos en {LANDING}")
